In [61]:

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("EmployeeRecordsSession") \
    .getOrCreate()

In [62]:
df = spark.read.csv("dataset.csv", header=True, inferSchema=True)
df.createOrReplaceTempView("orders")


In [63]:
df.show(50,truncate=False)

+---+-------+---------+---------+---------------+-------+---------------+
|ID | Name  |Amount($)|is Active|Order Date     |Sex    |Updated At     |
+---+-------+---------+---------+---------------+-------+---------------+
|152|Ånna   | 1 200   |Inactive |2/26/2023 22:12|FEMALE |6/1/2023 16:02 |
|415|Renée  |10.000,00|1        |1/15/2023 10:57|f      |2/13/2023 19:50|
|119|Judy   |1,000.50 |Active   |6/2/2023 17:14 |M      |4/6/2023 15:32 |
|365|Ivan   |-5000    |N        |2/2/2023 21:16 |f      |2/15/2023 21:07|
|30 |Bob    |NA       |TRUE     |NULL           |Other  |1/13/2023 1:20 |
|403|Frank  |500      |Active   |3/31/2023 0:58 |f      |4/6/2023 5:51  |
|64 |David  |2000     |NO       |1/16/2023 21:43|f      |6/4/2023 1:03  |
|482|Alice  |2.000,50 |Y        |6/10/2023 16:42|FEMALE |6/10/2023 20:17|
|442|Hannah |10.000,00|Inactive |2/8/2023 14:43 |F      |2/4/2023 7:50  |
|448|Renée  |2.000,50 |Active   |5/1/2023 17:50 |f      |1/22/2023 2:21 |
|31 |Bob    |2.000,50 |NO       |3/21/

In [64]:
df.printSchema(
)

root
 |-- ID : integer (nullable = true)
 |--  Name: string (nullable = true)
 |-- Amount($): string (nullable = true)
 |-- is Active: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Updated At: string (nullable = true)



In [66]:
#standardize col name
import re
new_cols=[]
for c in df.columns:
    nc=re.sub(r"[\s-]+","_",c.strip().lower())
    nc=re.sub(r"[^0-9a-z_]+","",nc)
    nc=re.sub(r"_+","_",nc).strip("_")
    new_cols.append(nc)

for old,new in zip(df.columns,new_cols):
    if old !=new:
        df=df.withColumnRenamed(old,new)
df.printSchema()


root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- is_active: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- updated_at: string (nullable = true)



In [67]:
from pyspark.sql import functions as F

In [68]:
#clean missing values and strings
na_tokens=["","na","n/a","null","-","--","unknown"]
for c,t in df.dtypes:
    if t=="string":
        df=df.withColumn(c,F.regexp_replace(F.col(c),"\xa0"," "))
        df=df.withColumn(c,F.trim(F.col(c)))
        df=df.withColumn(
            c,
            F.when(F.lower(F.col(c)).isin(na_tokens),None).otherwise(F.col(c))
        )
df.show(10,truncate=False)

+---+------+---------+---------+---------------+------+---------------+
|id |name  |amount   |is_active|order_date     |sex   |updated_at     |
+---+------+---------+---------+---------------+------+---------------+
|152|Ånna  |1 200    |Inactive |2/26/2023 22:12|FEMALE|6/1/2023 16:02 |
|415|Renée |10.000,00|1        |1/15/2023 10:57|f     |2/13/2023 19:50|
|119|Judy  |1,000.50 |Active   |6/2/2023 17:14 |M     |4/6/2023 15:32 |
|365|Ivan  |-5000    |N        |2/2/2023 21:16 |f     |2/15/2023 21:07|
|30 |Bob   |NULL     |TRUE     |NULL           |Other |1/13/2023 1:20 |
|403|Frank |500      |Active   |3/31/2023 0:58 |f     |4/6/2023 5:51  |
|64 |David |2000     |NO       |1/16/2023 21:43|f     |6/4/2023 1:03  |
|482|Alice |2.000,50 |Y        |6/10/2023 16:42|FEMALE|6/10/2023 20:17|
|442|Hannah|10.000,00|Inactive |2/8/2023 14:43 |F     |2/4/2023 7:50  |
|448|Renée |2.000,50 |Active   |5/1/2023 17:50 |f     |1/22/2023 2:21 |
+---+------+---------+---------+---------------+------+---------

In [69]:
#format amount col
df=df.withColumn("amount",F.regexp_replace("amount","０","0"))
df=df.withColumn("amount",F.regexp_replace("amount",r"[^0-9,\.-]",""))
df=df.withColumn("amount",F.regexp_replace("amount",r",","."))
df=df.withColumn("amount",F.regexp_replace("amount",r"\.",""))
df=df.withColumn("amount",F.col("amount").cast("double"))

In [70]:

df = df.withColumn(
    "is_active",
    F.when(F.lower("is_active").isin(["true", "t", "yes", "y", "1", "active"]), F.lit(True))
    .when(F.lower("is_active").isin(["false", "f", "no", "n", "0", "inactive"]), F.lit(False))
    .otherwise(F.lit(None).cast("boolean"))
)

In [71]:
df.show()

+---+-------+---------+---------+---------------+------+---------------+
| id|   name|   amount|is_active|     order_date|   sex|     updated_at|
+---+-------+---------+---------+---------------+------+---------------+
|152|   Ånna|   1200.0|    false|2/26/2023 22:12|FEMALE| 6/1/2023 16:02|
|415|  Renée|1000000.0|     true|1/15/2023 10:57|     f|2/13/2023 19:50|
|119|   Judy| 100050.0|     true| 6/2/2023 17:14|     M| 4/6/2023 15:32|
|365|   Ivan|  -5000.0|    false| 2/2/2023 21:16|     f|2/15/2023 21:07|
| 30|    Bob|     NULL|     true|           NULL| Other| 1/13/2023 1:20|
|403|  Frank|    500.0|     true| 3/31/2023 0:58|     f|  4/6/2023 5:51|
| 64|  David|   2000.0|    false|1/16/2023 21:43|     f|  6/4/2023 1:03|
|482|  Alice| 200050.0|     true|6/10/2023 16:42|FEMALE|6/10/2023 20:17|
|442| Hannah|1000000.0|    false| 2/8/2023 14:43|     F|  2/4/2023 7:50|
|448|  Renée| 200050.0|     true| 5/1/2023 17:50|     f| 1/22/2023 2:21|
| 31|    Bob| 200050.0|    false| 3/21/2023 9:17|  

In [72]:

# Dates

# order_date

from pyspark.sql.functions import col, trim, regexp_replace, when, coalesce, lit, expr,concat_ws

# Step 1: Trim spaces
df = df.withColumn("order_date", trim(col("order_date")))

# Step 2: Normalize separators (- or . -> /)
df = df.withColumn("order_date", regexp_replace(col("order_date"), "[-.]", "/"))

# Step 3: Fill missing time only if date exists
df = df.withColumn(
    "order_date",
    when(col("order_date").isNotNull() & (~col("order_date").rlike(r"\d{1,2}/\d{1,2}/\d{4}\s\d{1,2}:\d{2}")),
         concat_ws(" ", col("order_date"), lit("00:00")))
    .otherwise(col("order_date"))
)

In [73]:
df.show()

+---+-------+---------+---------+---------------+------+---------------+
| id|   name|   amount|is_active|     order_date|   sex|     updated_at|
+---+-------+---------+---------+---------------+------+---------------+
|152|   Ånna|   1200.0|    false|2/26/2023 22:12|FEMALE| 6/1/2023 16:02|
|415|  Renée|1000000.0|     true|1/15/2023 10:57|     f|2/13/2023 19:50|
|119|   Judy| 100050.0|     true| 6/2/2023 17:14|     M| 4/6/2023 15:32|
|365|   Ivan|  -5000.0|    false| 2/2/2023 21:16|     f|2/15/2023 21:07|
| 30|    Bob|     NULL|     true|           NULL| Other| 1/13/2023 1:20|
|403|  Frank|    500.0|     true| 3/31/2023 0:58|     f|  4/6/2023 5:51|
| 64|  David|   2000.0|    false|1/16/2023 21:43|     f|  6/4/2023 1:03|
|482|  Alice| 200050.0|     true|6/10/2023 16:42|FEMALE|6/10/2023 20:17|
|442| Hannah|1000000.0|    false| 2/8/2023 14:43|     F|  2/4/2023 7:50|
|448|  Renée| 200050.0|     true| 5/1/2023 17:50|     f| 1/22/2023 2:21|
| 31|    Bob| 200050.0|    false| 3/21/2023 9:17|  

In [81]:
from pyspark.sql.functions import col, when, upper, trim
df = df.withColumn(
    "sex",
    when(upper(trim(col("sex"))).isin("F", "FEMALE"), "F")
    .when(upper(trim(col("sex"))).isin("M", "MALE"), "M")
    .when(upper(trim(col("sex"))).isin("O", "OTHER"), "O")
    .otherwise(None)
)

df.show()

+---+-------+---------+---------+---------------+----+----------+
| id|   name|   amount|is_active|     order_date| sex|updated_at|
+---+-------+---------+---------+---------------+----+----------+
|152|   Ånna|   1200.0|    false|2/26/2023 22:12|   F|2023-06-01|
|415|  Renée|1000000.0|     true|1/15/2023 10:57|   F|2023-02-13|
|119|   Judy| 100050.0|     true| 6/2/2023 17:14|   M|2023-04-06|
|365|   Ivan|  -5000.0|    false| 2/2/2023 21:16|   F|2023-02-15|
| 30|    Bob|     NULL|     true|           NULL|   O|2023-01-13|
|403|  Frank|    500.0|     true| 3/31/2023 0:58|   F|2023-04-06|
| 64|  David|   2000.0|    false|1/16/2023 21:43|   F|2023-06-04|
|482|  Alice| 200050.0|     true|6/10/2023 16:42|   F|2023-06-10|
|442| Hannah|1000000.0|    false| 2/8/2023 14:43|   F|2023-02-04|
|448|  Renée| 200050.0|     true| 5/1/2023 17:50|   F|2023-01-22|
| 31|    Bob| 200050.0|    false| 3/21/2023 9:17|   O|2023-01-10|
|120|  Renée|   2000.0|    false| 2/16/2023 9:53|   M|2023-04-28|
|270|   Ån

In [76]:
df=df.withColumn("updated_at",trim(col("updated_at")))
df=df.withColumn("updated_at",regexp_replace(col("updated_at"),"[-.]","/"))

df=df.withColumn("updated_at",when(col("updated_at").isNotNull()
                &(~col("updated_at").rlike(r"\d{1,2}/\d{1,2}/\d{4}\s\d{1,2}:\d{2}"))
                ,concat_ws(" ",col("updated_at"),lit("00:00")))
                .otherwise(col("updated_at")))

In [77]:
df=df.withColumn(
    "updated_at",F.to_date("updated_at", "M/d/yyyy H:mm")
)

In [78]:
df.createOrReplaceTempView("querydf")


In [79]:
spark.sql("DESCRIBE querydf").show()

+----------+---------+-------+
|  col_name|data_type|comment|
+----------+---------+-------+
|        id|      int|   NULL|
|      name|   string|   NULL|
|    amount|   double|   NULL|
| is_active|  boolean|   NULL|
|order_date|   string|   NULL|
|       sex|   string|   NULL|
|updated_at|     date|   NULL|
+----------+---------+-------+



In [ ]:
spark.sql("""
    SELECT *,
           DENSE_RANK() OVER (
               ORDER BY updated_at DESC
           ) AS dense_rank
    FROM querydf
    
""").show()